# Training an Embedding Model From Scratch

Embedding models turn text into vectors where **similar meaning = nearby vectors**.

But how does it learn what's "similar"? This notebook builds and trains one from scratch.

---
## The Goal

```
embed("car problems")     → [0.8, 0.2, -0.1, ...]
embed("vehicle issues")   → [0.7, 0.3, -0.2, ...]  ← close! (similar meaning)
embed("pasta recipe")     → [-0.3, 0.9, 0.5, ...]  ← far! (different meaning)
```

The model must learn WITHOUT being told what words mean.
It learns entirely from **which texts appear together** (pairs).

---
## Training Data: Pairs of Similar Texts

The model learns from examples of texts that SHOULD have similar vectors:

```
Positive pair: ("How to fix a flat tire", "tire repair guide")
               → these should be CLOSE in vector space

Negative pair: ("How to fix a flat tire", "chocolate cake recipe")
               → these should be FAR apart
```

Where do these pairs come from?
- Search engines: (query, clicked result) = similar
- Q&A sites: (question, answer) = similar
- Paraphrases: (sentence, its rephrasing) = similar
- Random pairs from different documents = different

In [ ]:
import numpy as np
np.random.seed(42)

# Training data: pairs of similar sentences
# In reality, these come from search logs, Q&A sites, or paraphrase datasets

positive_pairs = [
    ("how to fix a car",          "vehicle repair guide"),
    ("how to fix a car",          "auto mechanic tips"),
    ("python programming",        "coding in python"),
    ("python programming",        "write python code"),
    ("best restaurants nearby",   "good places to eat"),
    ("best restaurants nearby",   "local food recommendations"),
    ("machine learning basics",   "intro to ML"),
    ("machine learning basics",   "learn AI fundamentals"),
    ("how to cook pasta",         "pasta recipe instructions"),
    ("how to cook pasta",         "making italian noodles"),
    ("stock market today",        "financial markets update"),
    ("stock market today",        "trading and investments"),
    ("weather forecast",          "temperature prediction"),
    ("weather forecast",          "will it rain tomorrow"),
    ("learn guitar",              "guitar lessons for beginners"),
    ("learn guitar",              "how to play guitar"),
]

print(f"Training Data: {len(positive_pairs)} positive pairs\n")
print(f"  Examples of SIMILAR pairs (should be close):")
for a, b in positive_pairs[:4]:
    print(f"    '{a}' ↔ '{b}'")

print(f"\n  Negative pairs (should be far) are created by random pairing:")
print(f"    '{positive_pairs[0][0]}' ↔ '{positive_pairs[4][1]}'  ← unrelated!")
print(f"    '{positive_pairs[2][0]}' ↔ '{positive_pairs[6][1]}'  ← unrelated!")

---
## Step 1: Vocabulary and Tokenization

First, we need to convert words to numbers. For this demo, we use simple word-level tokens.

In [ ]:
# Build vocabulary from all training texts
all_texts = []
for a, b in positive_pairs:
    all_texts.extend([a, b])

# Simple word tokenizer
all_words = set()
for text in all_texts:
    all_words.update(text.lower().split())

vocab = {"[PAD]": 0}  # padding token
for word in sorted(all_words):
    vocab[word] = len(vocab)

vocab_size = len(vocab)
print(f"Vocabulary: {vocab_size} words\n")
print(f"  Sample: {dict(list(vocab.items())[:10])}\n")

# Tokenize function
def tokenize(text, max_len=6):
    tokens = [vocab.get(w, 0) for w in text.lower().split()]
    # Pad or truncate to fixed length
    tokens = tokens[:max_len]
    tokens += [0] * (max_len - len(tokens))  # pad with 0
    return np.array(tokens)

# Demo
example = "how to fix a car"
tokens = tokenize(example)
print(f"  Tokenize '{example}':")
print(f"  → {tokens}")
print(f"  (each number = index in vocabulary)")

---
## Step 2: The Embedding Model Architecture

Our model:
1. Look up each token in an embedding table (random at first)
2. Average all token embeddings into ONE vector (the sentence embedding)

```
"how to fix a car"
   ↓ tokenize
[5, 12, 3, 1, 2]
   ↓ look up each in embedding table
[[0.2, -0.1, ...], [0.5, 0.3, ...], [0.1, 0.8, ...], ...]
   ↓ average all vectors
[0.3, 0.4, ...]  ← single vector representing the whole sentence
```

Real models (BERT, E5) use attention instead of averaging — but the principle is the same.

In [ ]:
# The embedding model
class EmbeddingModel:
    def __init__(self, vocab_size, embed_dim):
        # The ONLY learnable parameter: embedding table
        # Each word gets a random vector (will be trained)
        self.embeddings = np.random.randn(vocab_size, embed_dim) * 0.1
        self.embed_dim = embed_dim
        self.vocab_size = vocab_size
    
    def forward(self, token_ids):
        """Convert token IDs → single sentence vector."""
        # Look up embeddings for each token
        token_vectors = self.embeddings[token_ids]  # shape: (seq_len, embed_dim)
        
        # Average pooling (ignore padding tokens = 0)
        mask = (token_ids != 0).astype(float)
        if mask.sum() == 0:
            return np.zeros(self.embed_dim)
        
        # Weighted average (only non-padding tokens)
        sentence_vector = (token_vectors * mask[:, np.newaxis]).sum(axis=0) / mask.sum()
        
        # Normalize to unit length (for cosine similarity)
        norm = np.linalg.norm(sentence_vector)
        if norm > 0:
            sentence_vector = sentence_vector / norm
        
        return sentence_vector
    
    def embed(self, text):
        """Full pipeline: text → vector."""
        tokens = tokenize(text)
        return self.forward(tokens)

# Initialize model
embed_dim = 16  # real models: 768-1536
model = EmbeddingModel(vocab_size, embed_dim)

print(f"Embedding Model:")
print(f"  Vocabulary: {vocab_size} words")
print(f"  Embedding dim: {embed_dim}")
print(f"  Total parameters: {vocab_size * embed_dim:,} (just the embedding table!)\n")

# Test before training (random vectors — no meaning yet)
v1 = model.embed("how to fix a car")
v2 = model.embed("vehicle repair guide")
v3 = model.embed("pasta recipe instructions")

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

print(f"  BEFORE training (random — no meaning):")
print(f"    sim('fix car', 'vehicle repair') = {cosine_sim(v1, v2):.3f}  ← should be high")
print(f"    sim('fix car', 'pasta recipe')   = {cosine_sim(v1, v3):.3f}  ← should be low")
print(f"    (Both random — model hasn't learned anything yet)")

---
## Step 3: Contrastive Loss — The Training Signal

How does the model learn? Through **contrastive loss**:

```
For a positive pair (A, B):
  - Make embed(A) and embed(B) MORE similar     (pull together)
  - Make embed(A) and embed(random) LESS similar (push apart)
```

The most common approach: **InfoNCE loss** (used by OpenAI, Google, etc.)

```
Given a batch of N pairs: (a1,b1), (a2,b2), ..., (aN,bN)

For a1: b1 is the POSITIVE, b2...bN are NEGATIVES
Loss = -log( sim(a1,b1) / (sim(a1,b1) + sim(a1,b2) + ... + sim(a1,bN)) )

"Make the correct pair score highest among all options"
```

In [ ]:
# Contrastive loss explained with numbers
print("Contrastive Loss — How It Works:\n")

# Simulate a batch of 4 pairs
batch_a = ["how to fix a car", "python programming", "best restaurants nearby", "weather forecast"]
batch_b = ["vehicle repair guide", "coding in python", "good places to eat", "temperature prediction"]

print(f"  Batch of {len(batch_a)} pairs:\n")
for i, (a, b) in enumerate(zip(batch_a, batch_b)):
    print(f"    Pair {i}: '{a}' ↔ '{b}'")

# Compute similarity matrix
vecs_a = [model.embed(t) for t in batch_a]
vecs_b = [model.embed(t) for t in batch_b]

print(f"\n  Similarity matrix (each row = one query, columns = all candidates):")
print(f"  {'':20}", end="")
for i in range(4):
    print(f" b{i:>5}", end="")
print()

sim_matrix = np.zeros((4, 4))
for i in range(4):
    for j in range(4):
        sim_matrix[i, j] = cosine_sim(vecs_a[i], vecs_b[j])

for i in range(4):
    print(f"  a{i} '{batch_a[i][:16]:<16}'", end="")
    for j in range(4):
        marker = " ←" if i == j else "  "
        print(f" {sim_matrix[i,j]:>5.2f}{marker}", end="")
    print()

print(f"\n  Arrows show correct pairs (diagonal).")
print(f"  Goal: make diagonal values HIGHEST in each row.")
print(f"  Currently random — after training, diagonal will dominate.")

In [ ]:
# InfoNCE Loss implementation
print("InfoNCE Loss Function:\n")

def info_nce_loss(sim_matrix, temperature=0.1):
    """
    sim_matrix[i][j] = cosine similarity between a_i and b_j
    Correct pairs are on the diagonal (i==j)
    """
    n = sim_matrix.shape[0]
    # Scale by temperature (sharpens the distribution)
    logits = sim_matrix / temperature
    
    # For each row, correct answer is the diagonal element
    # Loss = -log(softmax(correct_pair))
    total_loss = 0
    for i in range(n):
        # Softmax: exp(correct) / sum(exp(all))
        exp_logits = np.exp(logits[i] - logits[i].max())  # subtract max for stability
        softmax = exp_logits / exp_logits.sum()
        # Loss for this example: -log(probability of correct pair)
        total_loss += -np.log(softmax[i] + 1e-8)
    
    return total_loss / n

loss = info_nce_loss(sim_matrix)
print(f"  Current loss (before training): {loss:.3f}")
print(f"  Perfect loss (diagonal = 1, rest = 0): ~0.0")
print(f"  Random loss (uniform): ~{np.log(4):.3f} = log(batch_size)")

print(f"\n  The loss says: 'How well can you pick the correct pair")
print(f"  out of all candidates in the batch?'")
print(f"\n  Temperature controls sharpness:")
print(f"    High temp (1.0): tolerant, soft probabilities")
print(f"    Low temp (0.05): strict, must clearly separate correct from wrong")

---
## Step 4: Training Loop

Now we train! The gradient updates the embedding table so that:
- Positive pairs get **pulled together**
- Negative pairs get **pushed apart**

In [ ]:
# Training the embedding model
print("Training the Embedding Model:\n")

def train_embedding_model(model, positive_pairs, epochs=300, lr=0.05, temperature=0.2):
    losses = []
    
    for epoch in range(epochs):
        # Shuffle pairs
        indices = np.random.permutation(len(positive_pairs))
        
        epoch_loss = 0
        # Process in batches of 4
        batch_size = 4
        n_batches = 0
        
        for batch_start in range(0, len(positive_pairs) - batch_size + 1, batch_size):
            batch_idx = indices[batch_start:batch_start + batch_size]
            batch = [positive_pairs[i] for i in batch_idx]
            
            # Tokenize batch
            tokens_a = [tokenize(a) for a, b in batch]
            tokens_b = [tokenize(b) for a, b in batch]
            
            # Forward: compute embeddings
            vecs_a = np.array([model.forward(t) for t in tokens_a])
            vecs_b = np.array([model.forward(t) for t in tokens_b])
            
            # Compute similarity matrix
            sim_matrix = vecs_a @ vecs_b.T  # dot product (vectors are normalized)
            
            # Compute loss
            loss = info_nce_loss(sim_matrix, temperature)
            epoch_loss += loss
            n_batches += 1
            
            # Gradient update (simplified: nudge embeddings directly)
            # For each pair, pull positive closer, push negatives apart
            for i in range(batch_size):
                # Positive: make a_i and b_i more similar
                for tok in tokens_a[i]:
                    if tok != 0:  # skip padding
                        model.embeddings[tok] += lr * vecs_b[i] / 6
                for tok in tokens_b[i]:
                    if tok != 0:
                        model.embeddings[tok] += lr * vecs_a[i] / 6
                
                # Negative: push a_i away from b_j (j != i)
                for j in range(batch_size):
                    if j != i:
                        for tok in tokens_a[i]:
                            if tok != 0:
                                model.embeddings[tok] -= lr * 0.3 * vecs_b[j] / 6
        
        avg_loss = epoch_loss / max(n_batches, 1)
        losses.append(avg_loss)
    
    return losses

# Train!
model = EmbeddingModel(vocab_size, embed_dim)
losses = train_embedding_model(model, positive_pairs, epochs=300, lr=0.03)

# Show training progress
print(f"  {'Epoch':<8} {'Loss':>8} {'Progress'}")
print(f"  {'─'*8} {'─'*8} {'─'*30}")
checkpoints = [0, 10, 30, 50, 100, 150, 200, 299]
for e in checkpoints:
    bar = '█' * int(max(0, (3.0 - losses[e]) * 10))
    print(f"  {e+1:<8} {losses[e]:>8.3f} {bar}")

print(f"\n  Loss decreased: {losses[0]:.3f} → {losses[-1]:.3f}")
print(f"  The model is learning to group similar texts together!")

In [ ]:
# Test AFTER training
print("Results AFTER Training:\n")

test_pairs = [
    ("how to fix a car", "vehicle repair guide", "similar"),
    ("how to fix a car", "auto mechanic tips", "similar"),
    ("how to fix a car", "pasta recipe instructions", "different"),
    ("how to fix a car", "stock market today", "different"),
    ("python programming", "coding in python", "similar"),
    ("python programming", "weather forecast", "different"),
    ("best restaurants nearby", "good places to eat", "similar"),
    ("best restaurants nearby", "learn guitar", "different"),
]

print(f"  {'Text A':<24} {'Text B':<24} {'Sim':>5} {'Expected':>9} {'Correct?'}")
print(f"  {'─'*24} {'─'*24} {'─'*5} {'─'*9} {'─'*8}")

correct = 0
for a, b, expected in test_pairs:
    va = model.embed(a)
    vb = model.embed(b)
    sim = cosine_sim(va, vb)
    
    predicted = "similar" if sim > 0.5 else "different"
    is_correct = predicted == expected
    correct += is_correct
    mark = "✓" if is_correct else "✗"
    
    print(f"  {a:<24} {b:<24} {sim:>5.2f} {expected:>9} {mark}")

print(f"\n  Accuracy: {correct}/{len(test_pairs)} ({correct/len(test_pairs)*100:.0f}%)")
print(f"\n  The model learned that:")
print(f"    'car' ≈ 'vehicle' ≈ 'auto' (same cluster)")
print(f"    'programming' ≈ 'coding' (same cluster)")
print(f"    'restaurants' ≈ 'places to eat' (same cluster)")
print(f"  Without EVER being told what these words mean!")

In [ ]:
# Visualize the learned embedding space
print("Learned Embedding Space (2D projection):\n")

# Embed all unique phrases
all_phrases = list(set([a for a, b in positive_pairs] + [b for a, b in positive_pairs]))
all_vecs = np.array([model.embed(p) for p in all_phrases])

# Simple 2D projection (take first 2 principal components)
# Center the data
centered = all_vecs - all_vecs.mean(axis=0)
# SVD for PCA
U, S, Vt = np.linalg.svd(centered, full_matrices=False)
projected = centered @ Vt[:2].T  # project to 2D

# Assign topics by keyword
def get_topic(phrase):
    if any(w in phrase for w in ['car', 'vehicle', 'auto', 'repair', 'mechanic', 'fix']):
        return 'cars'
    elif any(w in phrase for w in ['python', 'coding', 'code', 'programming']):
        return 'coding'
    elif any(w in phrase for w in ['restaurant', 'eat', 'food', 'places']):
        return 'food'
    elif any(w in phrase for w in ['weather', 'rain', 'temperature', 'forecast']):
        return 'weather'
    elif any(w in phrase for w in ['stock', 'market', 'financial', 'trading', 'investment']):
        return 'finance'
    elif any(w in phrase for w in ['guitar', 'play', 'lessons', 'learn guitar']):
        return 'music'
    elif any(w in phrase for w in ['machine', 'ML', 'AI', 'learning']):
        return 'ML'
    elif any(w in phrase for w in ['pasta', 'cook', 'recipe', 'noodle', 'italian']):
        return 'cooking'
    return 'other'

topics = [get_topic(p) for p in all_phrases]
topic_symbols = {'cars': '🚗', 'coding': '💻', 'food': '🍽️', 'weather': '🌤️',
                 'finance': '📈', 'music': '🎸', 'ML': '🤖', 'cooking': '🍝', 'other': '·'}

# ASCII scatter plot
width, height = 60, 20
grid = [[' ' for _ in range(width)] for _ in range(height)]

# Normalize to grid
x_min, x_max = projected[:, 0].min(), projected[:, 0].max()
y_min, y_max = projected[:, 1].min(), projected[:, 1].max()

placed = []
for i, (phrase, topic) in enumerate(zip(all_phrases, topics)):
    x = int((projected[i, 0] - x_min) / (x_max - x_min + 1e-8) * (width - 1))
    y = int((projected[i, 1] - y_min) / (y_max - y_min + 1e-8) * (height - 1))
    y = height - 1 - y  # flip y
    symbol = topic[0].upper()  # first letter of topic
    if 0 <= x < width and 0 <= y < height:
        grid[y][x] = symbol
        placed.append((x, y, phrase, topic))

print("  2D projection of learned embeddings:")
print("  (same letter = same topic cluster)\n")
for row in grid:
    print(f"  {''.join(row)}")

print(f"\n  Legend: C=cars, C=coding, F=food, W=weather, F=finance, M=music/ML, C=cooking")
print(f"\n  Similar topics cluster together in vector space!")
print(f"  This is what enables semantic search in RAG.")

---
## How Real Embedding Models Are Trained

Our demo is simplified. Real models (OpenAI `text-embedding-3`, Cohere `embed-v3`, `E5`, `BGE`):

| Aspect | Our Demo | Real Model |
|--------|----------|------------|
| Architecture | Average word vectors | Full transformer (BERT-like) |
| Parameters | ~1K | 100M-300M |
| Training data | 16 pairs | 100M-1B pairs |
| Embedding dim | 16 | 768-1536 |
| Batch size | 4 | 2048-65536 |
| Training time | Seconds | Days on many GPUs |

In [ ]:
# Real embedding model training details
print("Real Embedding Model Training:\n")

print("  1. ARCHITECTURE")
print("     Not just averaging — uses a full transformer:")
print("     Input → Tokenize → Transformer (12 layers) → [CLS] token → normalize")
print("     The [CLS] token embedding = the sentence embedding\n")

print("  2. TRAINING DATA SOURCES")
sources = [
    ("Search logs",      "(query, clicked doc)",    "100M+ pairs"),
    ("Q&A sites",        "(question, answer)",      "10M+ pairs"),
    ("Paraphrases",      "(sentence, rephrasing)",  "5M+ pairs"),
    ("NLI datasets",     "(premise, entailment)",   "1M+ pairs"),
    ("Title-abstract",   "(paper title, abstract)", "30M+ pairs"),
]
print(f"     {'Source':<20} {'Pair format':<25} {'Scale'}")
print(f"     {'─'*20} {'─'*25} {'─'*12}")
for src, fmt, scale in sources:
    print(f"     {src:<20} {fmt:<25} {scale}")

print(f"\n  3. HARD NEGATIVES (key to quality!)")
print(f"     Easy negative: 'car repair' vs 'chocolate cake' (obviously different)")
print(f"     Hard negative: 'car repair' vs 'car insurance'  (similar words, different intent!)")
print(f"     Training with hard negatives forces the model to learn MEANING, not just words.")

print(f"\n  4. MULTI-STAGE TRAINING")
print(f"     Stage 1: Pre-train on massive text (like LLM) — learn language")
print(f"     Stage 2: Fine-tune with contrastive loss on pairs — learn similarity")
print(f"     Stage 3: Fine-tune on hard negatives — sharpen distinctions")

In [ ]:
# Using pre-trained embedding models (what you'd actually do)
print("In Practice: You Don't Train Your Own (usually)\n")

print("  Just like you don't train your own LLM, you use a pre-trained embedding model:\n")

print("  Option 1: API (easiest)")
print("  ┌────────────────────────────────────────────────┐")
print("  │  import openai                                 │")
print("  │                                                │")
print("  │  response = openai.embeddings.create(          │")
print("  │      model='text-embedding-3-small',           │")
print("  │      input='how to fix a car'                  │")
print("  │  )                                             │")
print("  │  vector = response.data[0].embedding           │")
print("  │  # → [0.023, -0.041, 0.087, ...] (1536 dims)  │")
print("  └────────────────────────────────────────────────┘")

print("\n  Option 2: Local model (free, private)")
print("  ┌────────────────────────────────────────────────┐")
print("  │  from sentence_transformers import SentenceTransformer│")
print("  │                                                │")
print("  │  model = SentenceTransformer('all-MiniLM-L6-v2')│")
print("  │  vector = model.encode('how to fix a car')     │")
print("  │  # → [0.023, -0.041, ...] (384 dims)          │")
print("  └────────────────────────────────────────────────┘")

print("\n  Popular pre-trained embedding models:")
models = [
    ("OpenAI text-embedding-3-small", 1536, "$0.02/1M tokens", "API"),
    ("OpenAI text-embedding-3-large", 3072, "$0.13/1M tokens", "API"),
    ("Cohere embed-v3",               1024, "$0.10/1M tokens", "API"),
    ("all-MiniLM-L6-v2",              384,  "Free (local)",    "22M params"),
    ("E5-large-v2",                   1024, "Free (local)",    "335M params"),
    ("BGE-large-en",                  1024, "Free (local)",    "335M params"),
]
print(f"\n  {'Model':<32} {'Dims':>5} {'Cost':<16} {'Type'}")
print(f"  {'─'*32} {'─'*5} {'─'*16} {'─'*10}")
for name, dims, cost, typ in models:
    print(f"  {name:<32} {dims:>5} {cost:<16} {typ}")

---
## Summary: How Embeddings Enable RAG

```
Training (done by OpenAI/researchers, using millions of text pairs):
  Contrastive learning → model learns "similar text = similar vector"

Your RAG application (uses the pre-trained model):
  1. embed(each document chunk)      → store vectors
  2. embed(user question)            → query vector  
  3. find closest stored vectors     → relevant chunks
  4. give chunks to LLM             → answer
```

| Concept | Key Point |
|---------|----------|
| **Contrastive learning** | Pull similar pairs close, push different pairs apart |
| **InfoNCE loss** | "Pick the correct pair out of a batch" |
| **Hard negatives** | Similar-looking but different-meaning pairs (hardest to learn) |
| **Architecture** | Transformer that outputs one vector per input text |
| **In practice** | Use pre-trained model (API or local), don't train your own |
| **Training data** | Search logs, Q&A pairs, paraphrases — millions of pairs |